In [1]:
import joblib
print("joblib working")

joblib working


In [2]:
from pathlib import Path
import librosa
import numpy as np
import pandas as pd
import joblib

print("imports working")

imports working


In [3]:
from pathlib import Path

INPUT_AUDIO = Path("../input/sample.m4a")

print("Audio exists:", INPUT_AUDIO.exists())
print(INPUT_AUDIO)

Audio exists: True
..\input\sample.m4a


In [4]:
audio, sample_rate = librosa.load(INPUT_AUDIO, sr=16000)

print("Sample Rate:", sample_rate)
print("Duration:", round(len(audio)/sample_rate, 2), "seconds")

C:\Users\Sanjiv\AppData\Local\Temp\ipykernel_38236\2763852127.py:1: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sample_rate = librosa.load(INPUT_AUDIO, sr=16000)
c:\Users\Sanjiv\Desktop\SRM\PROJECTS\FluentIQ\venv\lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Sample Rate: 16000
Duration: 14.36 seconds


In [5]:
def extract_user_audio_features(audio_path):
    audio, sr = librosa.load(audio_path, sr=16000)

    duration = len(audio) / sr

    rms = librosa.feature.rms(y=audio)
    zcr = librosa.feature.zero_crossing_rate(audio)
    spectral_centroid = librosa.feature.spectral_centroid(y=audio, sr=sr)
    spectral_bandwidth = librosa.feature.spectral_bandwidth(y=audio, sr=sr)
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)

    features = {
        "duration_seconds": round(duration, 2),
        "word_count": 0,
        "words_per_minute": 0,
        "average_rms_energy": round(float(np.mean(rms)), 6),
        "average_zero_crossing_rate": round(float(np.mean(zcr)), 6),
        "spectral_centroid": round(float(np.mean(spectral_centroid)), 2),
        "spectral_bandwidth": round(float(np.mean(spectral_bandwidth)), 2)
    }

    mfcc_mean = np.mean(mfcc, axis=1)

    for i, value in enumerate(mfcc_mean):
        features[f"mfcc_{i+1}"] = round(float(value), 4)

    return features

In [6]:
user_features = extract_user_audio_features(INPUT_AUDIO)

pd.DataFrame([user_features]).T

C:\Users\Sanjiv\AppData\Local\Temp\ipykernel_38236\4220300164.py:2: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(audio_path, sr=16000)
c:\Users\Sanjiv\Desktop\SRM\PROJECTS\FluentIQ\venv\lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


,0
duration_seconds,14.360000
word_count,0.000000
words_per_minute,0.000000
average_rms_energy,0.123683
average_zero_crossing_rate,0.153414
spectral_centroid,1716.100000
spectral_bandwidth,1554.460000
mfcc_1,-260.057500
mfcc_2,83.968500
mfcc_3,7.997100


In [7]:
feature_columns = [
    "duration_seconds",
    "word_count",
    "words_per_minute",
    "average_rms_energy",
    "average_zero_crossing_rate",
    "spectral_centroid",
    "spectral_bandwidth",
    "mfcc_1","mfcc_2","mfcc_3","mfcc_4","mfcc_5",
    "mfcc_6","mfcc_7","mfcc_8","mfcc_9","mfcc_10",
    "mfcc_11","mfcc_12","mfcc_13"
]

X_user = pd.DataFrame([user_features])[feature_columns]

X_user

,duration_seconds,word_count,words_per_minute,average_rms_energy,average_zero_crossing_rate,spectral_centroid,spectral_bandwidth,mfcc_1,mfcc_2,mfcc_3,mfcc_4,mfcc_5,mfcc_6,mfcc_7,mfcc_8,mfcc_9,mfcc_10,mfcc_11,mfcc_12,mfcc_13
0,14.36,0,0,0.123683,0.153414,1716.1,1554.46,-260.0575,83.9685,7.9971,37.1869,-2.7935,-6.501,-22.2171,-26.4994,-10.409,-11.0195,-2.2226,-11.5357,0.1521


In [9]:
classifier = joblib.load("../models/speech_level_classifier.pkl")
regressor = joblib.load("../models/wpm_regressor.pkl")

print("Models loaded.")

Models loaded.


In [10]:
predicted_level = classifier.predict(X_user)[0]
predicted_score = regressor.predict(X_user)[0]

print("Predicted Speaking Level :", predicted_level)
print("Predicted Communication Score :", round(predicted_score, 2))

Predicted Speaking Level : Advanced
Predicted Communication Score : 74.07
